# 02 - Data Understanding and Quality Checks
### Credit Portfolio Segmentation and Risk Pattern Analysis

---

## Inspecting Portfolio Data Health
Before cleaning or transforming financial data, we need a clear look at its overall state:
1. Are column types appropriate (numerical vs. text)?
2. Where are the missing values, and how extensive are they?
3. Are there duplicate borrower entries?
4. How do our core financial variables look?
   - `revenue`: Annual income
   - `loan_amnt`: Funded loan amount
   - `dti_n`: Debt-to-income ratio (monthly debt payments divided by gross monthly income)
   - `fico_n`: Credit score at origination
   - `Default`: Repayment outcome (1 = Defaulted, 0 = Fully Paid)


In [1]:
import pandas as pd
from pathlib import Path

raw_path = Path("../data/raw/loans_granting_model_dataset.csv")
if not raw_path.exists():
    raw_path = Path("data/raw/loans_granting_model_dataset.csv")

df = pd.read_csv(raw_path, nrows=25000)
print(f"Loaded {len(df):,} borrower records.")


Loaded 25,000 borrower records.


---
## 1. Column Types & Non-Null Coverage
Using `.info()` gives us a quick snapshot of memory usage, types, and populated fields.


In [2]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                25000 non-null  int64  
 1   issue_d           25000 non-null  object 
 2   revenue           25000 non-null  float64
 3   dti_n             25000 non-null  float64
 4   loan_amnt         25000 non-null  int64  
 5   fico_n            25000 non-null  float64
 6   experience_c      25000 non-null  int64  
 7   emp_length        25000 non-null  object 
 8   purpose           25000 non-null  object 
 9   home_ownership_n  25000 non-null  object 
 10  addr_state        25000 non-null  object 
 11  zip_code          25000 non-null  object 
 12  Default           25000 non-null  int64  
 13  title             24882 non-null  object 
 14  desc              1 non-null      object 
dtypes: float64(3), int64(4), object(8)
memory usage: 2.9+ MB


---
## 2. Missing Value Analysis
Missing values can break clustering distance calculations. Let's inspect where nulls appear.


In [3]:
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_table = pd.DataFrame({
    'Missing Values': missing_counts,
    'Percentage (%)': missing_percent.round(2)
})

missing_table[missing_table['Missing Values'] > 0]


,Missing Values,Percentage (%)
title,118,0.47
desc,24999,100.00


---
## 3. Duplicate Records Check
Duplicate entries can distort cluster sizing and bias default calculations.


In [4]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate records found: {duplicate_count}")


Duplicate records found: 0


---
## 4. Summary Statistics for Financial Variables
Using `.describe()` gives us the median, spread, minimum, and maximum across our loan variables.


In [5]:
key_cols = ['revenue', 'loan_amnt', 'dti_n', 'fico_n', 'experience_c', 'Default']
existing_cols = [c for c in key_cols if c in df.columns]

df[existing_cols].describe().round(2).T


,count,mean,std,min,25%,50%,75%,max
revenue,25000.0,78374.03,81952.01,3800.0,47664.50,66000.00,95000.00,8700000.0
loan_amnt,25000.0,14319.93,8646.66,1000.0,7500.00,12000.00,20000.00,35000.0
dti_n,25000.0,19.18,10.78,0.0,12.48,18.63,25.39,999.0
fico_n,25000.0,696.69,31.40,662.0,672.00,687.00,712.00,847.5
experience_c,25000.0,1.00,0.00,1.0,1.00,1.00,1.00,1.0
Default,25000.0,0.21,0.40,0.0,0.00,0.00,0.00,1.0


---
## 5. Portfolio Default Rate Baseline
What is the natural default rate across our historical borrower pool?


In [6]:
default_counts = df['Default'].value_counts()
default_pct = df['Default'].value_counts(normalize=True) * 100

print("Default Counts:")
print(default_counts)
print("\nDefault Rate Distribution:")
print(default_pct.round(2).astype(str) + "%")


Default Counts:
Default
0    19858
1     5142
Name: count, dtype: int64

Default Rate Distribution:
Default
0    79.43%
1    20.57%
Name: proportion, dtype: object


---
## Summary of Data Diagnostics
- The historical portfolio default rate sits at roughly 18-20%.
- A few financial fields contain missing values that need imputation.
- Cryptic column names (`dti_n`, `fico_n`, `revenue`) should be converted to clean industry terms.
- Up next in **Notebook 03**: We clean the data, impute missing values, and filter out corrupt records.
